# Nykaa Domain Support Agent — Task 9: Structured Output Schema

## Objective
Define a JSON Schema that every final agent response must satisfy, then validate every response in code.

The schema standardizes:
- `conversation_id`
- `query`
- `intent`
- `answer`
- `sources`
- `order_details`
- `escalation`

The notebook exercises three response types:
1. A policy/RAG answer.
2. A successful order lookup.
3. An order query with a missing or unknown order ID.

The agent runs in deterministic `MOCK_LLM` mode. No paid API key is required.

## 1. Install and import dependencies

In [1]:
%pip install -U jsonschema langgraph chromadb sentence-transformers pandas numpy

  Attempting uninstall: jsonschema
    Found existing installation: jsonschema 4.25.1
    Uninstalling jsonschema-4.25.1:
      Successfully uninstalled jsonschema-4.25.1
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
from pathlib import Path
from typing import TypedDict, Optional, Literal
import re
import json

import pandas as pd
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
from jsonschema import Draft202012Validator
from langgraph.graph import StateGraph, START, END

PROJECT_DIR = Path.cwd()
CHROMA_DIR = PROJECT_DIR / "chroma_db"
SCORED_ORDERS_PATH = PROJECT_DIR / "orders_with_escalation_scores.csv"
ORDERS_PATH = PROJECT_DIR / "orders.csv"

if not CHROMA_DIR.exists():
    raise FileNotFoundError("chroma_db not found. Run Task 3 in the project directory.")

if SCORED_ORDERS_PATH.exists():
    ORDERS_INPUT = SCORED_ORDERS_PATH
elif ORDERS_PATH.exists():
    ORDERS_INPUT = ORDERS_PATH
else:
    raise FileNotFoundError("Place orders.csv or orders_with_escalation_scores.csv in the project directory.")

print("Project directory:", PROJECT_DIR.resolve())
print("Orders file:", ORDERS_INPUT.resolve())


Project directory: C:\Users\singh\Downloads
Orders file: C:\Users\singh\Downloads\orders_with_escalation_scores.csv


## 2. Load the selected RAG collection and order dataset

Sentence-based chunking was selected after Task 5 because it had higher mean Precision@3 and Recall@3 in the evaluation you ran.

In [3]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
COLLECTION_NAME = "nykaa_sentence_chunks"
TOP_K = 3

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
policy_collection = chroma_client.get_collection(COLLECTION_NAME)
embedding_model = SentenceTransformer(MODEL_NAME)

orders_df = pd.read_csv(ORDERS_INPUT)
required_columns = {
    "record_id", "status", "order_value_inr",
    "days_since_created", "delayed_shipment"
}
missing = required_columns - set(orders_df.columns)
if missing:
    raise ValueError(f"Orders dataset is missing columns: {sorted(missing)}")

orders_df["record_id"] = orders_df["record_id"].astype(str).str.strip()
orders_df["status"] = orders_df["status"].astype(str).str.strip()
orders_df["order_value_inr"] = pd.to_numeric(orders_df["order_value_inr"], errors="raise")
orders_df["days_since_created"] = pd.to_numeric(orders_df["days_since_created"], errors="raise")
orders_df["delayed_shipment"] = (
    orders_df["delayed_shipment"].astype(str).str.strip().str.lower()
    .map({"true": True, "false": False})
)
if orders_df["delayed_shipment"].isna().any():
    raise ValueError("delayed_shipment must contain only True/False.")
if orders_df["record_id"].duplicated().any():
    raise ValueError("Order record_id values must be unique.")

print("RAG collection:", COLLECTION_NAME, "| chunks:", policy_collection.count())
print("Orders loaded:", len(orders_df))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RAG collection: nykaa_sentence_chunks | chunks: 12
Orders loaded: 50


## 3. Recreate the Task 6 order lookup function

The score uses the same formula as revised Task 6:
`0.70 × delayed_signal + 0.30 × (days_since_created / 30)`

The threshold is derived from the 80th percentile of the composite score distribution.

In [4]:
DELAY_WEIGHT = 0.70
RECENCY_WEIGHT = 0.30
MAX_ORDER_AGE_DAYS = 30

orders_df["escalation_score"] = (
    DELAY_WEIGHT * orders_df["delayed_shipment"].astype(float)
    + RECENCY_WEIGHT * (orders_df["days_since_created"] / MAX_ORDER_AGE_DAYS)
).clip(0, 1)

ESCALATION_THRESHOLD = float(orders_df["escalation_score"].quantile(0.80))

def check_order_status(record_id: Optional[str]) -> dict:
    if not record_id or not str(record_id).strip():
        return {"found": False, "error": "Please provide your order ID."}

    requested_id = str(record_id).strip().upper()
    matches = orders_df[orders_df["record_id"].str.upper() == requested_id]

    if matches.empty:
        return {"found": False, "record_id": requested_id,
                "error": f"No order found for {requested_id}."}

    order = matches.iloc[0]
    score = round(float(order["escalation_score"]), 4)

    return {
        "found": True,
        "record_id": order["record_id"],
        "status": order["status"],
        "order_value_inr": float(order["order_value_inr"]),
        "escalation_score": score,
        "requires_escalation": score >= ESCALATION_THRESHOLD
    }

print("Task 6 lookup ready.")
print("Escalation threshold:", round(ESCALATION_THRESHOLD, 4))


Task 6 lookup ready.
Escalation threshold: 0.722


## 4. Define the JSON Schema

Every final response must contain exactly the seven top-level fields defined below. Conditional rules ensure:
- Policy responses have `intent = policy`, policy sources, and null order/escalation fields.
- Order responses have `intent = order`.
- A successful order lookup includes order details and escalation information.
- A missing/unknown order lookup has null order/escalation fields and a clear answer.

The schema is intentionally explicit so downstream applications can reliably parse the response.

In [5]:
RESPONSE_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "title": "NykaaSupportAgentResponse",
    "type": "object",
    "additionalProperties": False,
    "required": [
        "conversation_id", "query", "intent", "answer",
        "sources", "order_details", "escalation"
    ],
    "properties": {
        "conversation_id": {
            "type": "string",
            "minLength": 1
        },
        "query": {
            "type": "string",
            "minLength": 1
        },
        "intent": {
            "type": "string",
            "enum": ["policy", "order"]
        },
        "answer": {
            "type": "string",
            "minLength": 1
        },
        "sources": {
            "type": "array",
            "items": {"type": "string"},
            "uniqueItems": True
        },
        "order_details": {
            "oneOf": [
                {"type": "null"},
                {
                    "type": "object",
                    "additionalProperties": False,
                    "required": ["record_id", "status", "order_value_inr"],
                    "properties": {
                        "record_id": {"type": "string", "minLength": 1},
                        "status": {"type": "string", "minLength": 1},
                        "order_value_inr": {"type": "number", "minimum": 0}
                    }
                }
            ]
        },
        "escalation": {
            "oneOf": [
                {"type": "null"},
                {
                    "type": "object",
                    "additionalProperties": False,
                    "required": ["score", "threshold", "requires_escalation"],
                    "properties": {
                        "score": {"type": "number", "minimum": 0, "maximum": 1},
                        "threshold": {"type": "number", "minimum": 0, "maximum": 1},
                        "requires_escalation": {"type": "boolean"}
                    }
                }
            ]
        }
    },
    "allOf": [
        {
            "if": {"properties": {"intent": {"const": "policy"}}, "required": ["intent"]},
            "then": {
                "properties": {
                    "order_details": {"type": "null"},
                    "escalation": {"type": "null"}
                }
            }
        },
        {
            "if": {
                "properties": {
                    "intent": {"const": "order"},
                    "order_details": {"type": "object"}
                },
                "required": ["intent", "order_details"]
            },
            "then": {
                "properties": {
                    "escalation": {"type": "object"},
                    "sources": {"minItems": 1}
                }
            }
        },
        {
            "if": {
                "properties": {
                    "intent": {"const": "order"},
                    "order_details": {"type": "null"}
                },
                "required": ["intent", "order_details"]
            },
            "then": {
                "properties": {
                    "escalation": {"type": "null"}
                }
            }
        }
    ]
}

schema_validator = Draft202012Validator(RESPONSE_SCHEMA)
schema_errors = sorted(schema_validator.iter_errors({}), key=lambda e: list(e.path))
# Check the schema definition itself before validating agent responses.
from jsonschema import Draft202012Validator as SchemaMetaValidator
SchemaMetaValidator.check_schema(RESPONSE_SCHEMA)

print("PASS: JSON Schema definition is valid.")
print(json.dumps(RESPONSE_SCHEMA, indent=2))


PASS: JSON Schema definition is valid.
{
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "title": "NykaaSupportAgentResponse",
  "type": "object",
  "additionalProperties": false,
  "required": [
    "conversation_id",
    "query",
    "intent",
    "answer",
    "sources",
    "order_details",
    "escalation"
  ],
  "properties": {
    "conversation_id": {
      "type": "string",
      "minLength": 1
    },
    "query": {
      "type": "string",
      "minLength": 1
    },
    "intent": {
      "type": "string",
      "enum": [
        "policy",
        "order"
      ]
    },
    "answer": {
      "type": "string",
      "minLength": 1
    },
    "sources": {
      "type": "array",
      "items": {
        "type": "string"
      },
      "uniqueItems": true
    },
    "order_details": {
      "oneOf": [
        {
          "type": "null"
        },
        {
          "type": "object",
          "additionalProperties": false,
          "required": [
            "record

## 5. Implement RAG retrieval and query intent classification

In [6]:
ORDER_ID_PATTERN = re.compile(r"\bORD\d{4,}\b", flags=re.IGNORECASE)
ORDER_TERMS = [
    "order status", "track my order", "track order",
    "where is my order", "check my order", "order update"
]

def classify_query(query):
    match = ORDER_ID_PATTERN.search(query)
    lower = query.lower()
    if match or any(term in lower for term in ORDER_TERMS):
        return "order", match.group(0).upper() if match else None
    return "policy", None


def retrieve_policy(query):
    query_embedding = embedding_model.encode(
        [query], normalize_embeddings=True
    )[0].tolist()

    result = policy_collection.query(
        query_embeddings=[query_embedding],
        n_results=min(TOP_K, policy_collection.count()),
        include=["documents", "metadatas", "distances"]
    )

    contexts = []
    for doc, metadata, distance in zip(
        result.get("documents", [[]])[0],
        result.get("metadatas", [[]])[0],
        result.get("distances", [[]])[0]
    ):
        contexts.append({
            "text": doc,
            "source": metadata.get("source", "unknown"),
            "similarity": round(1 - float(distance), 4)
        })
    return contexts

print("Intent classifier and RAG retrieval function ready.")


Intent classifier and RAG retrieval function ready.


## 6. Build a LangGraph that emits schema-conforming responses

The final node constructs a response object. A dedicated validation node checks it against the JSON Schema. The graph returns only after validation succeeds; otherwise, validation raises an error rather than allowing a malformed response through.

In [7]:
class StructuredAgentState(TypedDict, total=False):
    conversation_id: str
    query: str
    intent: Literal["policy", "order"]
    record_id: Optional[str]
    contexts: list
    order_result: dict
    response: dict


def classify_node(state: StructuredAgentState) -> dict:
    intent, record_id = classify_query(state["query"])
    return {"intent": intent, "record_id": record_id}


def tool_node(state: StructuredAgentState) -> dict:
    if state["intent"] == "order":
        return {"order_result": check_order_status(state.get("record_id")), "contexts": []}
    return {"contexts": retrieve_policy(state["query"]), "order_result": {}}


def format_response_node(state: StructuredAgentState) -> dict:
    query = state["query"]
    conversation_id = state.get("conversation_id", "task9_demo")
    intent = state["intent"]

    if intent == "policy":
        contexts = state.get("contexts", [])
        if contexts:
            answer = (
                "According to the available project policy documents:\n\n"
                + "\n\n".join(
                    f"[{item['source']}] {item['text']}" for item in contexts
                )
            )
            sources = list(dict.fromkeys(
                item["source"] for item in contexts if item["source"] != "unknown"
            ))
        else:
            answer = "I don't know based on the available policy documents."
            sources = []

        response = {
            "conversation_id": conversation_id,
            "query": query,
            "intent": "policy",
            "answer": answer,
            "sources": sources,
            "order_details": None,
            "escalation": None
        }

    else:
        order_result = state.get("order_result", {})
        if not order_result.get("found"):
            answer = order_result.get("error", "Please provide a valid order ID.")
            response = {
                "conversation_id": conversation_id,
                "query": query,
                "intent": "order",
                "answer": answer,
                "sources": [],
                "order_details": None,
                "escalation": None
            }
        else:
            answer = (
                f"Order {order_result['record_id']} has status {order_result['status']}. "
                f"Order value: ₹{order_result['order_value_inr']:,.2f}. "
                f"Escalation score: {order_result['escalation_score']:.2f}. "
                + (
                    "This order is recommended for human review."
                    if order_result["requires_escalation"]
                    else "This order does not meet the project's escalation threshold."
                )
            )
            response = {
                "conversation_id": conversation_id,
                "query": query,
                "intent": "order",
                "answer": answer,
                "sources": ["orders dataset (synthetic project data)"],
                "order_details": {
                    "record_id": order_result["record_id"],
                    "status": order_result["status"],
                    "order_value_inr": order_result["order_value_inr"]
                },
                "escalation": {
                    "score": order_result["escalation_score"],
                    "threshold": round(ESCALATION_THRESHOLD, 4),
                    "requires_escalation": order_result["requires_escalation"]
                }
            }

    return {"response": response}


def validate_response_node(state: StructuredAgentState) -> dict:
    response = state["response"]
    errors = sorted(schema_validator.iter_errors(response), key=lambda e: list(e.path))

    if errors:
        error_messages = [
            f"{'/'.join(str(p) for p in error.path) or '<root>'}: {error.message}"
            for error in errors
        ]
        raise ValueError("Agent response failed JSON Schema validation: " + "; ".join(error_messages))

    print("[schema_validation] PASS — response conforms to JSON Schema")
    return {"response": response}


graph_builder = StateGraph(StructuredAgentState)
graph_builder.add_node("classify_intent", classify_node)
graph_builder.add_node("run_tool", tool_node)
graph_builder.add_node("format_response", format_response_node)
graph_builder.add_node("validate_response", validate_response_node)

graph_builder.add_edge(START, "classify_intent")
graph_builder.add_edge("classify_intent", "run_tool")
graph_builder.add_edge("run_tool", "format_response")
graph_builder.add_edge("format_response", "validate_response")
graph_builder.add_edge("validate_response", END)

structured_agent = graph_builder.compile()
print("PASS: Four-node structured-output LangGraph compiled.")


PASS: Four-node structured-output LangGraph compiled.


## 7. Test 1 — Policy question

The policy response must contain the required fields and set order-specific fields to null.

In [8]:
policy_response = structured_agent.invoke({
    "conversation_id": "task9_policy_test",
    "query": "How long does a cash-on-delivery refund take?"
})["response"]

print(json.dumps(policy_response, indent=2, ensure_ascii=False))
schema_validator.validate(policy_response)
assert policy_response["intent"] == "policy"
assert policy_response["order_details"] is None
assert policy_response["escalation"] is None
print("PASS: Policy response validated.")


[schema_validation] PASS — response conforms to JSON Schema
{
  "conversation_id": "task9_policy_test",
  "query": "How long does a cash-on-delivery refund take?",
  "intent": "policy",
  "answer": "According to the available project policy documents:\n\n[cod_refund.md] # Cash-on-Delivery (COD) Refund Timelines\n\nFor this simulated support environment, an approved COD refund is issued through the bank-account or UPI details collected during the refund process. The expected processing time is 5–7 business days after the returned item passes the required inspection. If the refund has not arrived after 7 business days, the customer should contact support with the order ID and refund reference, if available.\n\n[delivery_sla.md] # Delivery Service-Level Agreements\n\nFor this synthetic project, standard domestic orders have an estimated delivery window of 3–7 business days from shipment confirmation. Remote delivery locations may require up to 10 business days, and the estimate can change

## 8. Test 2 — Successful order lookup

Use an actual order ID from the loaded dataset. The response must include order details and the escalation object.

In [9]:
sample_order_id = str(orders_df.iloc[0]["record_id"])

order_response = structured_agent.invoke({
    "conversation_id": "task9_order_test",
    "query": f"Please check order {sample_order_id}."
})["response"]

print(json.dumps(order_response, indent=2, ensure_ascii=False))
schema_validator.validate(order_response)
assert order_response["intent"] == "order"
assert order_response["order_details"]["record_id"] == sample_order_id
assert 0 <= order_response["escalation"]["score"] <= 1
print("PASS: Successful order response validated.")


[schema_validation] PASS — response conforms to JSON Schema
{
  "conversation_id": "task9_order_test",
  "query": "Please check order ORD1001.",
  "intent": "order",
  "answer": "Order ORD1001 has status Delivered. Order value: ₹8,205.00. Escalation score: 0.00. This order does not meet the project's escalation threshold.",
  "sources": [
    "orders dataset (synthetic project data)"
  ],
  "order_details": {
    "record_id": "ORD1001",
    "status": "Delivered",
    "order_value_inr": 8205.0
  },
  "escalation": {
    "score": 0.0,
    "threshold": 0.722,
    "requires_escalation": false
  }
}
PASS: Successful order response validated.


## 9. Test 3 — Missing order ID

An order-status request without an ID must still conform to the schema, with null order and escalation objects and a request for the missing ID.

In [11]:
missing_id_response = structured_agent.invoke({
    "conversation_id": "task9_missing_id_test",
    "query": "Can you check my order status?"
})["response"]

print(json.dumps(missing_id_response, indent=2, ensure_ascii=False))
schema_validator.validate(missing_id_response)
assert missing_id_response["intent"] == "order"
assert missing_id_response["order_details"] is None
assert missing_id_response["escalation"] is None
assert "order id" in missing_id_response["answer"].lower()

print("PASS: Missing-ID response validated.")


[schema_validation] PASS — response conforms to JSON Schema
{
  "conversation_id": "task9_missing_id_test",
  "query": "Can you check my order status?",
  "intent": "order",
  "answer": "Please provide your order ID.",
  "sources": [],
  "order_details": null,
  "escalation": null
}
PASS: Missing-ID response validated.


## 10. Negative test — Deliberately malformed response must fail

A schema is useful only if invalid outputs are rejected. This test deliberately removes a required field and verifies that the validator catches the problem.

In [12]:
invalid_response = dict(policy_response)
invalid_response.pop("sources")

try:
    schema_validator.validate(invalid_response)
    raise AssertionError("Invalid response unexpectedly passed validation.")
except Exception as exc:
    from jsonschema import ValidationError
    if isinstance(exc, ValidationError):
        print("PASS: Invalid response was rejected.")
        print("Validation error:", exc.message)
    else:
        raise

# Also test an invalid escalation score outside the permitted [0, 1] range.
invalid_score_response = dict(order_response)
invalid_score_response["escalation"] = dict(order_response["escalation"])
invalid_score_response["escalation"]["score"] = 1.25

try:
    schema_validator.validate(invalid_score_response)
    raise AssertionError("Out-of-range escalation score unexpectedly passed.")
except ValidationError as exc:
    print("PASS: Escalation score outside [0, 1] was rejected.")
    print("Validation error:", exc.message)


PASS: Invalid response was rejected.
Validation error: 'sources' is a required property
PASS: Escalation score outside [0, 1] was rejected.
Validation error: {'score': 1.25, 'threshold': 0.722, 'requires_escalation': False} is not valid under any of the given schemas


## 11. Save the schema and validated test transcripts

The schema JSON and response transcript are saved as project evidence. The transcript contains only the three test responses produced above.

In [13]:
SCHEMA_PATH = PROJECT_DIR / "nykaa_response_schema.json"
TRANSCRIPT_PATH = PROJECT_DIR / "task9_structured_output_transcripts.json"

SCHEMA_PATH.write_text(
    json.dumps(RESPONSE_SCHEMA, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

transcript = {
    "task": "Task 9 — Structured Output Schema",
    "validation_library": "jsonschema Draft 2020-12",
    "responses": [
        policy_response,
        order_response,
        missing_id_response
    ],
    "negative_tests": {
        "missing_required_field_rejected": True,
        "escalation_score_above_one_rejected": True
    }
}

# Final validation of every response before saving.
for response in transcript["responses"]:
    schema_validator.validate(response)

TRANSCRIPT_PATH.write_text(
    json.dumps(transcript, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Schema saved:", SCHEMA_PATH.resolve())
print("Transcripts saved:", TRANSCRIPT_PATH.resolve())
print("\nPASS: All 3 agent responses conform to the JSON Schema.")
print("PASS: Both malformed-response tests were rejected.")
print("PASS: Task 9 structured output validation completed.")


Schema saved: C:\Users\singh\Downloads\nykaa_response_schema.json
Transcripts saved: C:\Users\singh\Downloads\task9_structured_output_transcripts.json

PASS: All 3 agent responses conform to the JSON Schema.
PASS: Both malformed-response tests were rejected.
PASS: Task 9 structured output validation completed.
